## Exercício 13 

In [ ]:

import numpy as np
from si.neural_networks.activation import TanhActivation, SoftmaxActivation

# Dados de exemplo (batch)
X = np.array([[1.0, 2.0, 3.0],
              [0.5, -1.0, 2.0]])

# Tanh
tanh = TanhActivation()
tanh.set_input_shape((3,))
tanh_out = tanh.forward_propagation(X)
print("Tanh output:\n", tanh_out)

tanh_grad = tanh.backward_propagation(np.ones_like(tanh_out))
print("Tanh backward:\n", tanh_grad)

# Softmax
softmax = SoftmaxActivation()
softmax.set_input_shape((3,))
softmax_out = softmax.forward_propagation(X)
print("Softmax output (rows sum to 1):\n", softmax_out)
print("Row sums:", softmax_out.sum(axis=1))

softmax_grad = softmax.backward_propagation(np.ones_like(softmax_out))
print("Softmax backward:\n", softmax_grad)


## Exercício 14 

In [ ]:

import os
import numpy as np

from datasets import DATASETS_PATH
from si.io.data_file import read_data_file
from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.layers import DenseLayer
from si.neural_networks.activation import TanhActivation, SoftmaxActivation
from si.neural_networks.losses import CategoricalCrossEntropy
from si.neural_networks.optimizers import SGD

# Load iris dataset
csv_file = os.path.join(DATASETS_PATH, "iris.csv")
ds = read_data_file(filename=csv_file, label=True, sep=",")

X = ds.X
y_labels = ds.y

# One-hot encoding
classes = np.unique(y_labels)
class_to_idx = {c: i for i, c in enumerate(classes)}
y_idx = np.array([class_to_idx[v] for v in y_labels])
y = np.zeros((len(y_idx), len(classes)))
y[np.arange(len(y_idx)), y_idx] = 1.0

# Build network
nn = NeuralNetwork(
    layers=[
        DenseLayer(input_size=4, output_size=8),
        TanhActivation(),
        DenseLayer(input_size=8, output_size=3),
        SoftmaxActivation(),
    ],
    loss=CategoricalCrossEntropy(),
    optimizer=SGD(learning_rate=0.01)
)

# Train
history = nn.fit(X, y, epochs=20, batch_size=16, verbose=False)

print("Final training loss:", history["loss"][-1])


## Exercício 15 

In [ ]:

import os
import numpy as np

from datasets import DATASETS_PATH
from si.io.data_file import read_data_file
from si.neural_networks.optimizers import Adam

# Load breast_bin dataset
csv_file = os.path.join(DATASETS_PATH, "breast_bin", "breast-bin.csv")
ds = read_data_file(filename=csv_file, label=True, sep=",")

X = ds.X.astype(float)
y = ds.y.astype(float)

# Inicialização simples de pesos
rng = np.random.default_rng(42)
w = rng.normal(scale=0.01, size=X.shape[1])
b = 0.0

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def loss(y_true, y_prob):
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))

opt_w = Adam(learning_rate=0.01)
opt_b = Adam(learning_rate=0.01)

# Loss inicial
p = sigmoid(X @ w + b)
loss0 = loss(y, p)

# Algumas iterações
for _ in range(5):
    p = sigmoid(X @ w + b)
    dz = p - y
    grad_w = X.T @ dz / X.shape[0]
    grad_b = np.mean(dz)

    w = opt_w.update(w, grad_w)
    b = float(opt_b.update(np.array(b), np.array(grad_b)))

# Loss final
p_new = sigmoid(X @ w + b)
loss1 = loss(y, p_new)

print("Initial loss:", loss0)
print("Final loss:", loss1)
